# Boosted Decision Tree

In [ ]:
import pandas as pd
import numpy as np
import re
import math
import time
from datetime import timedelta
from pathlib import Path
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (
    f1_score, make_scorer, roc_auc_score, accuracy_score, balanced_accuracy_score
)

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    'duke_lesions_radiomic' : FILE_PATH / 'duke_lesions_radiomic_medsam.csv',
    'duke_lesions' : FILE_PATH / 'duke_lesions.csv',
    'ambl_lesions_radiomic' : FILE_PATH / 'ambl_lesions_radiomic_medsam.csv',
    'ambl_lesions' : FILE_PATH / 'ambl_lesions.csv'
}

In [2]:
def best_threshold(y_true, y_prob):
    """
    Sceglie la soglia che massimizza F1 macro sul TRAIN fold
    """
    thresholds = np.linspace(0.05, 0.95, 50)
    scores = [
        f1_score(y_true, (y_prob >= t).astype(int),
                 average="macro", zero_division=0)
        for t in thresholds
    ]
    return thresholds[np.argmax(scores)]

# Training Duke


In [3]:
def training_duke(file_path: Path, csv_name: str):
    df = pd.read_csv(file_path)

    if "Patient ID" not in df.columns:
        raise ValueError(f"{csv_name} - manca Patient ID")

    # ================= TARGET =================
    df["ER_class"]   = pd.to_numeric(df["ER"], errors="coerce")
    df["PR_class"]   = pd.to_numeric(df["PR"], errors="coerce")
    df["HER2_class"] = pd.to_numeric(df["HER2"], errors="coerce")

    targets = ["ER_class", "PR_class", "HER2_class"]
    df = df.dropna(subset=targets).copy()
    for t in targets:
        df[t] = df[t].astype(int)

    # rimuovo target con una sola classe
    targets = [t for t in targets if df[t].nunique() > 1]
    if not targets:
        return None

    # ================= FEATURES =================
    drop_cols = [
        "Patient ID", "lesion idx", "tumor/benign",
        "GRADE", "isTN", "Breast",
        "ER", "PR", "HER2"
    ] + targets

    X = df.drop(columns=drop_cols, errors="ignore")
    y = df[targets]
    groups = df["Patient ID"]

    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.mean(numeric_only=True))
    X.columns = [re.sub(r"\[|\]|<", "", c) for c in X.columns]

    # ================= STRATIFIED GROUP CV =================
    y_strat = y["HER2_class"].astype(str)
    n_splits = min(5, y_strat.value_counts().min())
    if n_splits < 2:
        return None

    sgkf = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )
    splits = list(sgkf.split(X, y_strat, groups))


    #Debug
    """print("\n[CHECK] Distribuzione HER2 per fold (AMBL)")
    for k, (tr, te) in enumerate(splits):
        tr_counts = y.iloc[tr]["HER2_class"].value_counts(normalize=True)
        te_counts = y.iloc[te]["HER2_class"].value_counts(normalize=True)
        print(f"Fold {k}")
        print(" Train:", tr_counts.to_dict())
        print(" Test :", te_counts.to_dict())"""

    # ================= GRID SEARCH =================
    base_model = HistGradientBoostingClassifier(
        random_state=42,
        class_weight="balanced",
        early_stopping=False
    )

    def multi_f1(y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        return np.mean([
            f1_score(y_true[:, i], y_pred[:, i],
                        average="macro", zero_division=0)
            for i in range(y_true.shape[1])
        ])

    grid = GridSearchCV(
        MultiOutputClassifier(base_model),
        param_grid={
            "estimator__learning_rate": [0.05, 0.1],
            "estimator__max_iter": [100, 200],
            "estimator__max_depth": [3, 5],
        },
        scoring=make_scorer(multi_f1),
        cv=splits,
        n_jobs=-1,
        error_score="raise"
    )

    grid.fit(X, y)

    best_params = {
        k.replace("estimator__", ""): v
        for k, v in grid.best_params_.items()
    }

    # ================= CV EVALUATION =================
    fold_reports = []

    for fold_id, (tr, te) in enumerate(splits):

        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        model = MultiOutputClassifier(
            HistGradientBoostingClassifier(
                **best_params,
                random_state=42,
                class_weight="balanced",
                early_stopping=False
            )
        )

        model.fit(X_tr, y_tr)

        proba_tr = model.predict_proba(X_tr)
        proba_te = model.predict_proba(X_te)

        fold_metrics = {}

        for i, col in enumerate(targets):

            # === BEST THRESHOLD SOLO SU TRAIN ===
            t_opt = best_threshold(
                y_tr.iloc[:, i].values,
                proba_tr[i][:, 1]
            )

            y_pred = (proba_te[i][:, 1] >= t_opt).astype(int)

            fold_metrics[col] = {
                "f1": f1_score(
                    y_te.iloc[:, i],
                    y_pred,
                    average="macro",
                    zero_division=0
                ),
                "accuracy": accuracy_score(
                    y_te.iloc[:, i],
                    y_pred
                ),
                "balanced_accuracy": balanced_accuracy_score(y_te.iloc[:, i], y_pred),
                "auc": roc_auc_score(
                    y_te.iloc[:, i],
                    proba_te[i][:, 1]
                )
            }

        fold_reports.append(fold_metrics)

    return {
        "best_params": best_params,
        "mean_cv_score": grid.best_score_,
        "std_cv_score": grid.cv_results_["std_test_score"][grid.best_index_],
        "fold_reports": fold_reports,
        "targets_used": targets,
        "cv_results": grid.cv_results_
    }

# Training AMBL


In [4]:
def training_ambl(file_path: Path, csv_name: str):
    df = pd.read_csv(file_path)

    if "Patient ID" not in df.columns:
        raise ValueError(f"{csv_name} - manca 'Patient ID' necessario per Group split")

    df_validi = df.copy()

    # --- Controllo colonne target attese ---
    required_targets = ["ER [SII]", "PR [SII]", "HER2 [SII]"]
    missing = [c for c in required_targets if c not in df_validi.columns]
    if missing:
        print(f"[ERRORE] {csv_name} - mancano colonne target: {missing}")
        return None

    # --- Creo i target binari direttamente (già binari nel Duke) ---
    final_target_list = ["ER_class", "PR_class", "HER2_class"]

    df_validi["ER_class"]   = pd.to_numeric(df_validi["ER [SII]"], errors="coerce")
    df_validi["PR_class"]   = pd.to_numeric(df_validi["PR [SII]"], errors="coerce")
    df_validi["HER2_class"] = pd.to_numeric(df_validi["HER2 [SII]"], errors="coerce")

    # Tengo solo righe con tutti i target presenti e casto a int
    df_validi = df_validi.dropna(subset=final_target_list).copy()
    for col in final_target_list:
        df_validi[col] = df_validi[col].astype(int)

    # --- Tolgo target con 1 sola classe ---
    targets_da_rimuovere = []
    for col in final_target_list:
        if df_validi[col].nunique() < 2:
            print(f"[ATTENZIONE] {csv_name} - Target {col} ha una sola classe. Lo escludo.")
            targets_da_rimuovere.append(col)

    for col in targets_da_rimuovere:
        final_target_list.remove(col)

    if len(final_target_list) == 0:
        print(f"[ERRORE] {csv_name} - Nessun target valido (>=2 classi).")
        return None

    # --- Features / Target / Groups ---
    raw_target_cols = ["ER", "PR", "HER2"]

    features_to_drop = [
        "Patient ID", "lesion idx", "tumor/benign", "GRADE", "isTN", "Breast"
    ] + raw_target_cols + final_target_list

    features = df_validi.drop(columns=features_to_drop, errors="ignore")
    target = df_validi[final_target_list]
    groups = df_validi["Patient ID"]

    # Imputazione features numeriche
    features = features.apply(pd.to_numeric, errors="coerce")
    features = features.fillna(features.mean(numeric_only=True))

    # Pulizia nomi colonne
    features.columns = [re.sub(r"\[|\]|<", "", col) for col in features.columns]

    # --- StratifiedGroupKFold ---
    if "HER2_class" in final_target_list:
        y_strat = target["HER2_class"].astype(str)
        n_pos = int(target["HER2_class"].sum())
        n_splits = 3 if n_pos < 10 else 5
    else:
        y_strat = target.astype(int).astype(str).agg("_".join, axis=1)
        n_splits = 5


    # Debug
    min_class_count = y_strat.value_counts().min()
    n_splits = min(n_splits, min_class_count)

    if n_splits < 2:
        print(f"[ERRORE] {csv_name} - troppo pochi campioni per CV stratificata")
        return None

    vc = y_strat.value_counts()
    rare = vc[vc < n_splits].index
    if len(rare) > 0:
        y_strat = y_strat.where(~y_strat.isin(rare), other="RARE")

    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = list(sgkf.split(features, y_strat, groups=groups))

    """print("\n[CHECK] Distribuzione HER2 per fold (AMBL)")
    for k, (tr, te) in enumerate(splits):
        tr_counts = target.iloc[tr]["HER2_class"].value_counts(normalize=True)
        te_counts = target.iloc[te]["HER2_class"].value_counts(normalize=True)
        print(f"Fold {k}")
        print(" Train:", tr_counts.to_dict())
        print(" Test :", te_counts.to_dict())"""

    base_model = HistGradientBoostingClassifier(
        random_state=42,
        class_weight="balanced",
        early_stopping=False,
        l2_regularization=0.0,
        min_samples_leaf=10
    )
    multi_output_model = MultiOutputClassifier(base_model)

    iperparametri = {
        "estimator__learning_rate": [0.05, 0.1],
        "estimator__max_iter": [100, 200],
        "estimator__max_depth": [3, 5],
    }

    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            scores.append(f1_score(y_true[:, i], y_pred[:, i], average="macro", zero_division=0))
        return float(np.mean(scores))

    scorer = make_scorer(multi_f1_scorer)

    total_combinations = math.prod(len(v) for v in iperparametri.values())
    #print(f"\nInizio Grid Search HistGradientBoosting (GRID MINIMAL: {total_combinations} combinazioni) per: {csv_name}")

    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=splits,
        scoring=scorer,
        n_jobs=-1,
        verbose=1,
        refit=True,
        return_train_score=False,
        error_score="raise"
    )




    grid_search.fit(features, target)


    # Stampo return_train_score
    #print(pd.DataFrame(grid_search.cv_results_))


    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    clean_best_params = {k.replace("estimator__", ""): v for k, v in best_params.items()}

    final_params = {
        "random_state": 42,
        "class_weight": "balanced",
        "early_stopping": False,
        "l2_regularization": 0.0,
        "min_samples_leaf": 10,
        **clean_best_params
    }

    # Metriche per fold
    fold_reports = []
    for k, (train_idx, test_idx) in enumerate(splits):
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        model_clone = MultiOutputClassifier(HistGradientBoostingClassifier(**final_params))
        model_clone.fit(X_train, y_train)

        y_pred = model_clone.predict(X_test)


        y_proba_list = model_clone.predict_proba(X_test)

        # DEBUG: Fold k
        """print(f"\n[DEBUG] Fold {k}")
        for i, col in enumerate(final_target_list):
            proba_i = y_proba_list[i]
            y_true_i = y_test.iloc[:, i].values

            print(
                f"  Target: {col} | "
                f"y_true classes: {np.unique(y_true_i)} | "
                f"proba shape: {proba_i.shape}"
            )"""




        fold_metrics = {}
        for i, col in enumerate(final_target_list):
            y_true_i = y_test.iloc[:, i].values
            y_pred_i = y_pred[:, i]

            f1 = f1_score(y_true_i, y_pred_i, average="macro", zero_division=0)
            acc = accuracy_score(y_true_i, y_pred_i)
            bal_acc = balanced_accuracy_score(y_true_i, y_pred_i)

            auc_val = np.nan
            proba_i = y_proba_list[i]

            # Caso BINARIO
            if len(np.unique(y_true_i)) == 2 and proba_i.shape[1] == 2:
                auc_val = roc_auc_score(y_true_i, proba_i[:, 1])

            # Caso MULTICLASSE (AMBL)
            elif len(np.unique(y_true_i)) > 2:
                try:
                    auc_val = roc_auc_score(
                        y_true_i,
                        proba_i,
                        multi_class="ovr",
                        average="macro"
                    )
                except ValueError:
                    auc_val = np.nan

            fold_metrics[col] = {
                'f1': f1,
                'accuracy': acc,
                'balanced_accuracy': bal_acc,
                'auc': auc_val
            }
        # Debug
        """print(f"\nMetriche Fold {k}")
        for col, m in fold_metrics.items():
            auc_str = "nan" if np.isnan(m["auc"]) else f"{m['auc']:.3f}"
            print(
                f"  {col}: "
                f"F1={m['f1']:.3f} | "
                f"ACC={m['accuracy']:.3f} | "
                f"AUC={auc_str}"
            )"""
            
        fold_reports.append(fold_metrics)

    final_result = {
        **clean_best_params,
        "mean_score": best_score,
        "std_score": grid_search.cv_results_["std_test_score"][grid_search.best_index_],
        "fold_reports": fold_reports,
        "targets_used": final_target_list,
        "n_splits_used": n_splits,
        "n_rows_used": int(len(df_validi)),
        "cv_results": grid_search.cv_results_ 
    }

    return final_result

# Vado a stampare gli output in una maniera piú leggibile

In [5]:
def print_grid_search_results(results_per_dataset, save_csv=True, output_path="BoostedDecisionTree.csv"):
    print("\n" + "=" * 80)
    print(" " * 20 + "Metriche (MEDIA ± STD) per target")
    print("=" * 80)

    rows = []

    for dataset_name, best_result in results_per_dataset.items():
        if best_result is None:
            continue

        fold_reports = best_result["fold_reports"]
        target_names = best_result.get("targets_used", [])

        print(f"\n\nDataset: {dataset_name}")
        print("-" * 80)

        for target_name in target_names:

            f1_list  = np.array([fold[target_name]["f1"] for fold in fold_reports], dtype=float)
            acc_list = np.array([fold[target_name]["accuracy"] for fold in fold_reports], dtype=float)
            auc_list = np.array([fold[target_name]["auc"] for fold in fold_reports], dtype=float)
            bal_list = np.array([fold[target_name]["balanced_accuracy"] for fold in fold_reports], dtype=float)

            f1_mean,  f1_std  = np.mean(f1_list),  np.std(f1_list)
            acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)
            bal_mean, bal_std = np.mean(bal_list), np.std(bal_list)

            valid_auc = ~np.isnan(auc_list)
            auc_mean = np.mean(auc_list[valid_auc]) if valid_auc.any() else np.nan
            auc_std  = np.std(auc_list[valid_auc])  if valid_auc.any() else np.nan

            # ===== STAMPA =====
            print(f"\nTarget: {target_name}")
            print(f"  F1-score           = {f1_mean:.3f}  ±  {f1_std:.3f}")
            print(f"  Accuracy           = {acc_mean:.3f}  ±  {acc_std:.3f}")
            print(f"  Balanced Accuracy  = {bal_mean:.3f}  ±  {bal_std:.3f}")
            print(
                f"  AUC                = {auc_mean:.3f}  ±  {auc_std:.3f}"
                if not np.isnan(auc_mean)
                else f"  AUC                = NaN     ±  NaN"
            )

            # ===== CSV =====
            rows.append({
                "dataset": dataset_name,
                "target": target_name,
                "F1-score": f"{f1_mean:.3f} ± {f1_std:.3f}",
                "Accuracy": f"{acc_mean:.3f} ± {acc_std:.3f}",
                "Balanced Accuracy": f"{bal_mean:.3f} ± {bal_std:.3f}",
                "AUC": (
                    f"{auc_mean:.3f} ± {auc_std:.3f}"
                    if not np.isnan(auc_mean)
                    else "NaN ± NaN"
                )
            })

    # ===== SALVATAGGIO FILE =====
    if save_csv and rows:
        df_out = pd.DataFrame(rows)
        output_path = Path(output_path)
        df_out.to_csv(output_path, index=False)
        print(f"\n Risultati salvati in: {output_path.resolve()}")


# Lettura dei file

In [6]:
start_time = time.time()

# Eseguo il training per tutti i dataset
results_per_dataset = {}

for name, file_path in datasets.items():

    name_lower = name.lower()

    if "ambl" in name_lower:
        results_per_dataset[name] = training_ambl(file_path, name)

    elif "duke" in name_lower:
        results_per_dataset[name] = training_duke(file_path, name)

    else:
        raise ValueError(f"Dataset non riconosciuto: {name}")
    
# Stampa risultati
print_grid_search_results(results_per_dataset)

end_time = time.time()

# Tempo totale
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")


Fitting 2 folds for each of 8 candidates, totalling 16 fits
Fitting 2 folds for each of 8 candidates, totalling 16 fits

                    Metriche (MEDIA ± STD) per target


Dataset: duke_lesions_radiomic
--------------------------------------------------------------------------------

Target: ER_class
  F1-score           = 0.500  ±  0.063
  Accuracy           = 0.570  ±  0.052
  Balanced Accuracy  = 0.530  ±  0.061
  AUC                = 0.504  ±  0.099

Target: PR_class
  F1-score           = 0.503  ±  0.037
  Accuracy           = 0.522  ±  0.019
  Balanced Accuracy  = 0.545  ±  0.049
  AUC                = 0.556  ±  0.084

Target: HER2_class
  F1-score           = 0.428  ±  0.060
  Accuracy           = 0.477  ±  0.073
  Balanced Accuracy  = 0.435  ±  0.060
  AUC                = 0.430  ±  0.043


Dataset: duke_lesions
--------------------------------------------------------------------------------

Target: ER_class
  F1-score           = 0.523  ±  0.061
  Accuracy           = 0.